# Open Library｜作品・作家・版の記録を調べる
**技術講習会「文学研究のためのデジタル資料活用」｜2026年10月16日**

Open Libraryでは、作家の登録情報を**Author**、作品をまとめる登録情報を**Work**、個々の版・刊行物の登録情報を**Edition**と呼びます。

このNotebookでは、**Workと、それに紐づくEditionの登録状況**を調べます。Editionの登録件数は、販売部数や読者数ではありません。

**セルの左側の ▶ を上から順番に押して実行します。** 「すべてのセルを実行」は必要ありません。

- **前半**：Joseph Conradの8作品。保存済みの参考値を表示します（講習会用APIへの通信なし）。
- **後半**：作品を検索し、**Work IDを1つ選んで**、年代別Edition登録件数・個別Editionの書誌情報・関連Work・登録言語を調べます。

出版年代と件数は**2026年2月28日版の保存データ**、個別Editionの詳細は**Open Library公開APIの現在の記録**を使用します（取得時点が異なるため件数・版の対応が違う場合があります）。


## 1. Joseph Conradの8作品を比較する

まず、あらかじめ保存してある8作品について、WorkごとのEdition登録件数を表示します。

ここにあるのは**Open Libraryの公開ページで確認して記録した参考値（2026年10月14日）**です。後半の自由検索で使う**2026年2月28日版のデータ**とは件数が異なる場合があります。

作品は初出・初刊年の順に並べています。比べているのは各WorkのEdition登録件数であり、その年に出版された本の数ではありません。

次のセルを実行しても、講習会用APIへの通信は発生しません。

In [ ]:
# @title Conradの8作品を表で表示する（サーバーへの通信なし） {display-mode: "form"}
import pandas as pd
from IPython.display import display

# Open Library公開ページで確認した参考値（2026年10月14日）。
# このセルはインターネットや講習会用APIにアクセスしません。
CONRAD_EIGHT = [
    {"work_id": "OL38630W", "title": "Almayer's Folly", "editions": 97, "year": 1895},
    {"work_id": "OL38815W", "title": 'The Nigger of the "Narcissus"', "editions": 93, "year": 1897},
    {"work_id": "OL38663W", "title": "Heart of Darkness", "editions": 553, "year": 1899},
    {"work_id": "OL38684W", "title": "Lord Jim", "editions": 345, "year": 1900},
    {"work_id": "OL38700W", "title": "Nostromo", "editions": 255, "year": 1904},
    {"work_id": "OL39108W", "title": "The Secret Agent", "editions": 335, "year": 1907},
    {"work_id": "OL38763W", "title": "Victory", "editions": 146, "year": 1915},
    {"work_id": "OL38827W", "title": "The Shadow-Line", "editions": 147, "year": 1917},
]

conrad_df = pd.DataFrame(CONRAD_EIGHT).sort_values("year").reset_index(drop=True)
display(conrad_df.rename(columns={
    "year": "初出・初刊年の目安",
    "title": "作品名",
    "work_id": "Work ID",
    "editions": "Edition登録件数（参考値）",
})[["初出・初刊年の目安", "作品名", "Work ID", "Edition登録件数（参考値）"]])

In [ ]:
# @title Conradの8作品をグラフで表示する（サーバーへの通信なし） {display-mode: "form"}
import matplotlib.pyplot as plt

plot_df = conrad_df.iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 6))
labels = [f'{r.year}  {r.title}' for r in plot_df.itertuples()]
bars = ax.barh(labels, plot_df["editions"])
ax.bar_label(bars, padding=3)
ax.set_xlim(0, max(plot_df["editions"]) * 1.15)
ax.set_xlabel("Edition records in Open Library (saved reference values)")
ax.set_title("Joseph Conrad — 8 Work records")
fig.tight_layout()
plt.show()
print("データ：Open Library公開ページの保存済み参考値（2026年10月14日）")

## 2. 作品を探す

作家名と作品名を入力して、次のコードセルを実行します。**表示された候補から、調べたい作品のWork IDを1つ選んでください。** Author IDを選ぶ必要はありません。

作品名の検索は、サーバー側で著者による絞り込みをしていません。候補表では入力した作家名・タイトルとの一致を優先して並べますが、**上位だから正しいWorkというわけではありません**。同名の別作品、同じ作品に対応する複数のWork、年の誤登録もあります。必要に応じてOpen Libraryのリンクから詳細を確認してください。


In [ ]:
# @title 作家名と作品名からWork候補を探す（APIに接続） {display-mode: "form"}
AUTHOR_NAME = "夏目漱石" # @param {type:"string"}
WORK_TITLE = "吾輩は猫である" # @param {type:"string"}

# 以下は変更不要です
API_URL = "https://bunsekisan.jp/ol-workshop-api"

import time
import unicodedata
import requests
import pandas as pd
from IPython.display import display

def norm(value):
    return unicodedata.normalize("NFKC", str(value or "")).casefold().strip()

def api_get(path, params):
    # 混雑時の再試行は1回だけ
    for attempt in range(2):
        try:
            response = requests.get(API_URL + path, params=params, timeout=(5, 18))
            if response.status_code in (429, 503) and attempt == 0:
                time.sleep(3)
                continue
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == 1:
                raise
            time.sleep(3)
    raise RuntimeError("検索サーバーに接続できません")

def author_display(work):
    # 入力作家名と一致する登録名を先頭にする（原著者・翻訳者の役割判定ではありません）。
    names = [str(x).strip() for x in (work.get("author_names") or []) if str(x).strip()]
    names = sorted(names, key=lambda name: norm(name) != norm(AUTHOR_NAME))
    shown = names[:3]
    return "、".join(shown) + (" ほか" if len(names) > 3 else "") if shown else "—"

def show_year(value):
    return str(value) if value is not None and str(value).strip() else "—"

# 再検索時に古いWorkや集計結果を誤って使わないよう初期化します。
author_candidates = []
work_candidates = []
work_candidates_by_id = {}
result = None
api_df = pd.DataFrame(columns=["work_id", "title", "linked_edition_records"])
language_df = pd.DataFrame(columns=["lang", "label", "edition_records"])

try:
    a = api_get("/v1/authors", {"name": AUTHOR_NAME, "limit": 12})
    w = api_get("/v1/works", {"title": WORK_TITLE, "limit": 30})
    author_candidates = a.get("candidates", [])
    work_candidates = w.get("works", [])

    # 入力した作家名、作品タイトルの一致を優先して候補を並べ替えます。
    # サーバーから返された最大30候補の表示順だけが変わります。
    def matches_author(work):
        return any(norm(name) == norm(AUTHOR_NAME)
                   for name in (work.get("author_names") or []))

    def matches_title(work):
        found, asked = norm(work.get("title")), norm(WORK_TITLE)
        return found == asked or any(found.startswith(asked + s)
                                     for s in (" ", ",", ":", ";"))

    work_candidates.sort(key=lambda x: (
        not matches_author(x), not matches_title(x),
        -int(x.get("linked_edition_records") or 0), x.get("work_id", "")
    ))
    work_candidates_by_id = {x["work_id"]: x for x in work_candidates}
    print("検索結果：Open Library 2026年2月28日版の保存データ")
except (requests.RequestException, RuntimeError, ValueError, KeyError) as exc:
    author_candidates, work_candidates, work_candidates_by_id = [], [], {}
    print("検索できませんでした：", exc)
    print("少し待ってこのセルだけ再実行してください。")

print("\n【作品の候補】（次のセルではWork IDだけ入力します）")
if work_candidates:
    display(pd.DataFrame([{
        "Work ID": x["work_id"],
        "タイトル": x["title"],
        "著者（登録情報）": author_display(x),
        "Work記録年": show_year(x.get("work_first_year")),
        "最古Edition年": show_year(x.get("earliest_linked_edition_year")),
        "Edition登録件数": x.get("linked_edition_records", 0),
        "Open Library": f'https://openlibrary.org/works/{x["work_id"]}'
    } for x in work_candidates[:15]]).style.hide(axis="index"))
    print("この表から調べたいWork IDを1つ控えてください。Author IDは自動的に参照します。")
    print("Work記録年と最古Edition年は異なることがあります。どちらも歴史的な初刊年を保証しません。")
    print("同じ作品が複数のWorkに分かれる場合があるため、件数の多さだけで同定できません。")
else:
    print("候補を表示できませんでした。作品名を変えて検索してください。")


## 3. 選んだWorkのEdition登録状況と出版年代を調べる

前の作品候補表から**Work IDを1つ**選び、次の入力欄に貼り付けます。初期設定は夏目漱石『吾輩は猫である』（`OL1174971W`）です。Author IDの手入力は必要ありません。

このセルでは、指定したWorkのEdition登録総件数と、**出版年の記録を読み取れたEditionの年代別件数**をまとめて表示します。次のセルで年代別グラフを描きます。集計はOpen Library **2026年2月28日版**の保存データによります。

記録上の空白は、実際にその年代に出版されなかったことを意味しません。出版年の欠損や、別のWork IDへの分散もあります。関連Workの比較や年代範囲指定は後ろの独立した節で行います。


In [ ]:
# @title 選んだWorkの版数・出版年代をまとめて調べる（APIに接続） {display-mode: "form"}
WORK_ID = "OL1174971W" # @param {type:"string"}

# 前の検索結果で選んだWorkを使います。Author IDの手入力は不要です。
import pandas as pd
result = None
selected_comparison_author_id = None
selected_author_message = ""
selected_work = None
api_df = pd.DataFrame(columns=["work_id", "title", "linked_edition_records"])
language_df = pd.DataFrame(columns=["lang", "label", "edition_records"])
edition_decade_df = pd.DataFrame(columns=["decade", "edition_records"])
try:
    if "work_candidates_by_id" not in globals() or not work_candidates_by_id:
        raise ValueError("最初に『作家名と作品名からWork候補を探す』を実行してください。")
    selected_id = WORK_ID.strip()
    if selected_id not in work_candidates_by_id:
        raise ValueError(f"{selected_id} は直前の検索候補にありません。先に作品名を検索し直してください。")

    candidate = work_candidates_by_id[selected_id]
    linked_ids = list(dict.fromkeys(candidate.get("author_ids") or []))
    if not linked_ids:
        raise ValueError("このWorkには登録Author IDがありません。別のWork候補を確認してください。")

    linked_set = set(linked_ids)
    exact_matches = [x for x in author_candidates
                     if x.get("author_id") in linked_set and norm(x.get("name")) == norm(AUTHOR_NAME)]
    linked_matches = [x for x in author_candidates if x.get("author_id") in linked_set]
    if exact_matches:
        selected_comparison_author_id = exact_matches[0]["author_id"]
        selected_author_message = "入力した作家名に一致し、このWorkにも紐づくAuthor ID"
        if len(exact_matches) > 1:
            selected_author_message += "（複数候補の先頭を暫定採用）"
    elif linked_matches:
        selected_comparison_author_id = linked_matches[0]["author_id"]
        selected_author_message = "このWorkに紐づき、作家検索でも見つかったAuthor ID（完全一致ではありません）"
    elif len(linked_ids) == 1:
        selected_comparison_author_id = linked_ids[0]
        selected_author_message = "このWorkに登録された唯一のAuthor ID（名前の一致は未確認）"
    else:
        selected_comparison_author_id = sorted(linked_ids)[0]
        selected_author_message = "複数Author IDから暫定選択（入力作家名との一致は未確認）"

    result = api_get("/v1/collections/by-authors", {
        "author_ids": selected_comparison_author_id,
        "target_work_id": selected_id,
        "display_limit": 15,
    })
    selected_work = result.get("statistics_full_matching_set", {}).get("target")
    if not selected_work:
        raise ValueError("保存データで指定Workの情報を確認できませんでした。")
    print("【Open Library保存データ：2026年2月28日版】")
    print(f'作品：{selected_work["title"]}（{selected_work["work_id"]}）')
    print(f'Edition登録件数：{int(selected_work["linked_edition_records"]):,}件')
    print('このWorkの登録Author ID：' + '、'.join(selected_work.get("registered_author_ids") or ["記録なし"]))
    print('作品ページ：https://openlibrary.org/works/' + selected_work["work_id"])
    print("\n【このWorkの出版年代別Edition登録件数】")
    buckets = selected_work.get("edition_decade_counts")
    if buckets is None:
        print("年代別の項目を取得できませんでした。APIの追加項目 edition_decade_counts を確認してください。")
    else:
        by_decade = {}
        for item in buckets:
            try:
                decade = int(item["decade"])
                count = int(item["edition_records"])
            except (KeyError, TypeError, ValueError):
                continue
            if 1000 <= decade <= 2090 and count >= 0:
                by_decade[decade] = by_decade.get(decade, 0) + count
        if not by_decade:
            print("出版年を読み取れるEditionの登録は見つかりませんでした。")
        else:
            edition_decade_df = pd.DataFrame([
                {"decade": decade, "edition_records": by_decade.get(decade, 0)}
                for decade in range(min(by_decade), max(by_decade) + 1, 10)
            ])
            display(edition_decade_df.rename(columns={
                "decade": "年代（年）", "edition_records": "Edition登録件数"
            }).style.hide(axis="index"))
            counted = sum(by_decade.values())
            total = int(selected_work["linked_edition_records"])
            print(f"出版年を集計できたEdition：{counted:,}件／Work全体：{total:,}件")
            if counted < total:
                print("差分には出版年を読み取れない版などが含まれます。")
            elif counted > total:
                print("年代別計上と全体件数に差があります。元の書誌レコードの確認が必要です。")
    print("関連Workの比較・年代範囲指定は後ろの独立した節で扱います。")
except (requests.RequestException, RuntimeError, KeyError, ValueError, TypeError) as exc:
    result = None
    selected_work = None
    print("取得できませんでした：", exc)
    print("Work IDを確認してください。混雑時はこのセルだけ再実行してください。")


In [ ]:
# @title 選んだWorkのEdition登録件数を年代別グラフにする（通信なし） {display-mode: "form"}
import plotly.express as px
from IPython.display import display, HTML

if "edition_decade_df" not in globals() or edition_decade_df.empty:
    print("先に「選んだWorkの版数・出版年代をまとめて調べる」を実行してください。")
else:
    # Plotlyはブラウザー上で描画するため、日本語の作品名にも対応しやすくなります。
    title = selected_work.get("title", "") if selected_work else ""
    fig = px.bar(edition_decade_df, x="decade", y="edition_records",
                 labels={"decade": "年代（10年単位）", "edition_records": "Edition登録件数"},
                 title=f"{title} — Editions by decade")
    fig.update_traces(hovertemplate="%{x}年代: %{y}件<extra></extra>")
    fig.update_layout(height=380,
                      font=dict(family='Arial, "Noto Sans CJK JP", "Yu Gothic", Meiryo, sans-serif'),
                      margin=dict(l=40, r=30, t=65, b=50))
    fig.update_xaxes(tickmode="array", tickvals=edition_decade_df["decade"].tolist(), tickangle=-45)
    fig.show()
    print("注意：2026-02-28保存データのEdition登録年です。出版活動の空白を確定するものではありません。")


## 4. 個々のEditionの書誌情報を見る（参考）

Editionごとに**出版年月・出版社・出版地・版の表示・書誌データの取り込み元**などを確認します。こちらは講習会用データベースではなく、**Open Libraryの公開APIにある現在の記録**から、一度に最大20件だけ取得します。<https://openlibrary.org/dev/docs/api/books>

そのため、上の2026年2月28日版の件数・年代別グラフと、取得できるEditionや版情報が異なる場合があります。取得した20件は全Editionを代表する標本ではなく、**この表から出版年代の推移を計算することはしません**。

「第何版」は必ず登録されているとは限りません。また、`source_records` はMARC等の書誌データの取り込み元を示し、**所蔵館の一覧ではありません**。


In [ ]:
# @title 個別Editionの出版年・出版社・書誌データの由来を表示する（Open Library公開APIに接続） {display-mode: "form"}
EDITION_SAMPLE_LIMIT = 10 # @param {type:"integer"}
EDITION_OFFSET = 0 # @param {type:"integer"}

import re
import requests
import pandas as pd
from IPython.display import display

edition_details_df = pd.DataFrame()

def short_join(items, max_items=3, max_characters=150):
    if isinstance(items, str):
        items = [items]
    if not isinstance(items, list):
        return "—"
    texts = []
    for item in items[:max_items]:
        if isinstance(item, dict):
            item = item.get("name") or item.get("key") or str(item)
        if item is not None and str(item).strip():
            texts.append(str(item).strip())
    if not texts:
        return "—"
    value = "、".join(texts)
    if len(items) > max_items:
        value += f" ほか{len(items)-max_items}件"
    return value if len(value) <= max_characters else value[:max_characters-1] + "…"

try:
    if not isinstance(result, dict) or not selected_work:
        raise ValueError("先に『選んだWorkの版数・出版年代』を取得してください。")
    if not (1 <= int(EDITION_SAMPLE_LIMIT) <= 20):
        raise ValueError("取得件数は1～20件にしてください。")
    if not (0 <= int(EDITION_OFFSET) <= 1000):
        raise ValueError("取得開始位置は0～1000にしてください。")
    wid = selected_work["work_id"]
    url = f"https://openlibrary.org/works/{wid}/editions.json"
    response = requests.get(url,
                            params={"limit": int(EDITION_SAMPLE_LIMIT), "offset": int(EDITION_OFFSET)},
                            timeout=(5, 18),
                            headers={"User-Agent": "LiteraryMetadataWorkshop/1.0 (non-commercial educational example)"})
    response.raise_for_status()
    data = response.json()
    entries = data.get("entries", [])
    if not isinstance(entries, list):
        raise ValueError("Open Libraryから予想外の応答が返されました。")
    for ed in entries:
        if not isinstance(ed, dict):
            continue
        eid = str(ed.get("key") or "").rsplit("/", 1)[-1]
        if not re.fullmatch(r"OL[0-9]+M", eid):
            continue
        edition_details_df.loc[len(edition_details_df), [
            "Edition ID", "タイトル", "出版年・日付（登録値）", "版の表示", "出版社",
            "出版地", "言語", "書誌データの取り込み元", "Open Library"
        ]] = [
            eid,
            str(ed.get("title") or "—"),
            str(ed.get("publish_date") or "—"),
            str(ed.get("edition_name") or "—"),
            short_join(ed.get("publishers") or []),
            short_join(ed.get("publish_places") or []),
            short_join(ed.get("languages") or []),
            short_join(ed.get("source_records") or [], max_characters=170),
            f"https://openlibrary.org/books/{eid}"
        ]
    print("【Open Library公開APIの現在の記録（2026-02-28保存データとは別）】")
    print(f'Work：{wid}／今回取得したEdition：{len(edition_details_df)}件（取得開始位置：{EDITION_OFFSET}）')
    if edition_details_df.empty:
        print("表示できるEditionがありませんでした。")
    else:
        display(edition_details_df.style.hide(axis="index"))
        print("元の記録を詳しく見る場合は、各EditionのURLを開いてください。")
        print("版の表示が『—』なら、第何版かを判断できる情報が登録されていません。")
except (requests.RequestException, RuntimeError, KeyError, ValueError, TypeError) as exc:
    edition_details_df = pd.DataFrame()
    print("個別Editionの取得に失敗しました：", exc)
    print("Open Library本体の公開APIは講習会用サーバーとは別です。失敗しても後続の関連Work・言語別比較を続けられます。")


## 5. 参考：関連するWorkを比較する

選んだWorkに直接紐づく**Author IDを比較用に自動選択**し、そのAuthor IDに登録されたWorkのEdition件数を比較します。Author IDごとの記録集合であり、同名作家の全著作ではありません。

年代を指定すると、**比較するWork集合**を絞り込めます。年代はWorkの登録値（なければ関連Editionの最古年）で判断し、各WorkのEdition件数は全期間の登録件数です。年代を指定しなければ、上で取得した集計結果を再利用します。


In [ ]:
# @title 関連Workの比較表（年代・表示件数を指定） {display-mode: "form"}
YEAR_FROM = "" # @param {type:"string"}
YEAR_TO = "" # @param {type:"string"}
DISPLAY_LIMIT = 15 # @param {type:"integer"}

api_df = pd.DataFrame(columns=["work_id", "title", "linked_edition_records"])
author_comparison_result = None
try:
    if not isinstance(result, dict) or not selected_work or not selected_comparison_author_id:
        raise ValueError("先に『選んだWorkの版数・出版年代』を取得してください。")
    if not (1 <= int(DISPLAY_LIMIT) <= 50):
        raise ValueError("表示件数は1～50にしてください。")
    if bool(YEAR_FROM.strip()) != bool(YEAR_TO.strip()):
        raise ValueError("年代指定は開始年と終了年を両方入力してください。")
    if YEAR_FROM.strip() and (not YEAR_FROM.isdecimal() or not YEAR_TO.isdecimal()
                              or int(YEAR_FROM) > int(YEAR_TO)):
        raise ValueError("年代は開始・終了の順に数字で指定してください。")
    if not YEAR_FROM.strip() and int(DISPLAY_LIMIT) == 15:
        author_comparison_result = result  # API通信を増やさず、同じ結果を使う
    else:
        params = {
            "author_ids": selected_comparison_author_id,
            "target_work_id": selected_work["work_id"],
            "display_limit": int(DISPLAY_LIMIT),
        }
        if YEAR_FROM.strip():
            params.update({"year_from": int(YEAR_FROM), "year_to": int(YEAR_TO)})
        author_comparison_result = api_get("/v1/collections/by-authors", params)

    stat = author_comparison_result["statistics_full_matching_set"]
    api_df = pd.DataFrame(author_comparison_result.get("works_displayed", []))
    print("【2026年2月28日版：このAuthor IDに紐づくWork】")
    print("比較用Author ID：", selected_comparison_author_id)
    print("選択方法：", selected_author_message)
    print(f'比較集合：{stat["work_records"]:,} Workレコード')
    print(f'Edition登録件数の中央値：{stat["median_linked_edition_records"]}件')
    target = stat.get("target")
    if target and stat.get("target_in_set"):
        print(f'選択Workの順位：{target["rank_competition"]:,}位／{target["rank_denominator_work_records"]:,} Workレコード')
    else:
        print("選択Workはこの比較集合に含まれないため、順位は付けません。")
    print(f'\n【表示するWork：{len(api_df)}件】')
    if api_df.empty:
        print("該当するWorkはありません。")
    else:
        display(api_df[["work_id", "title", "linked_edition_records"]].rename(columns={
            "work_id": "Work ID", "title": "作品名", "linked_edition_records": "Edition登録件数"
        }).style.hide(axis="index"))
    print("中央値・順位は表示件数ではなく、条件に一致するWork全体を対象に計算します。")
except (requests.RequestException, RuntimeError, KeyError, ValueError, TypeError) as exc:
    author_comparison_result = None
    api_df = pd.DataFrame(columns=["work_id", "title", "linked_edition_records"])
    print("比較集合を取得できませんでした：", exc)
    print("年代や表示件数を確認し、必要ならこのセルを再実行してください。")


In [ ]:
# @title 関連WorkのEdition登録件数をグラフにする（通信なし） {display-mode: "form"}
import plotly.express as px

if "api_df" not in globals() or api_df.empty:
    print("グラフのデータがありません。先に『関連Workの比較表』を実行してください。")
else:
    plot_api = api_df.iloc[::-1].copy()
    plot_api["作品表示"] = [
        f'{row.title} [{row.work_id}]' for row in plot_api.itertuples()
    ]
    fig = px.bar(plot_api, x="linked_edition_records", y="作品表示", orientation="h",
                 text="linked_edition_records",
                 labels={"linked_edition_records": "Edition登録件数", "作品表示": "Work"},
                 title=f"Related Works — Author ID {selected_comparison_author_id}")
    fig.update_layout(height=max(380, 38 * len(plot_api) + 130),
                      font=dict(family='Arial, "Noto Sans CJK JP", "Yu Gothic", Meiryo, sans-serif'),
                      margin=dict(l=20, r=65, t=65, b=50))
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.show()
    print("日本語の書名はブラウザー側の日本語フォントを利用して表示します。")
    print("このグラフは、選んだAuthor IDに紐づくWorkの登録件数の比較です。")


## 6. 選んだWorkの版を言語別に調べる

選択した**Work ID**について、登録言語ごとのEdition登録件数を一覧で表示します。追加のAPI通信は発生しません。途中のAuthor比較で年代を指定しても、このWorkの言語別件数は変わりません。

非英語作品の場合、英語で登録されたEditionは英語への翻訳・出版を調べる手掛かりになります。ただし、**英語版の登録件数が翻訳出版の総数を意味するわけではありません**。言語タグがないEditionもあります。


In [ ]:
# @title 選んだWorkの言語別Edition登録件数を一覧で表示する（通信なし） {display-mode: "form"}
import pandas as pd
from IPython.display import display

LANGUAGE_NAMES = {
    "eng": "English", "jpn": "Japanese", "fre": "French",
    "ger": "German", "spa": "Spanish", "ita": "Italian",
    "rus": "Russian", "chi": "Chinese", "kor": "Korean",
    "por": "Portuguese", "ara": "Arabic",
    "NO_LANGUAGE_TAG": "言語未登録",
}

if "result" not in globals() or not isinstance(result, dict):
    print("先に『選んだWorkの版数・出版年代をまとめて調べる』を実行してください。")
else:
    language_target = result.get("statistics_full_matching_set", {}).get("target")
    if not language_target:
        print("指定したWork IDのデータが見つかりません。")
    else:
        language_counts = language_target.get("edition_language_counts")
        if language_counts is None:
            print("言語別件数がAPIから返されていません。")
        else:
            counts = {row["lang"]: int(row["edition_records"]) for row in language_counts}
            print(f'【{language_target["title"]}／{language_target["work_id"]}】')
            print(f'Work全体のEdition登録件数：{int(language_target["linked_edition_records"]):,}件')
            if counts:
                language_df = pd.DataFrame([
                    {"言語コード": code,
                     "言語": LANGUAGE_NAMES.get(code, code),
                     "Edition登録件数": number}
                    for code, number in sorted(counts.items(), key=lambda x: (-x[1], x[0]))
                ])
                print("\n【登録言語別のEdition件数（上位15件）】")
                display(language_df.head(15).style.hide(axis="index"))
            else:
                print("言語別のEdition登録情報がありません。")
            print("言語未登録の版や複数言語で登録された版があるため、言語別件数の合計はWork全体と一致しない場合があります。")


---
**数値について**　Edition登録件数は、出版部数・販売部数・読者数・文学的評価を表すものではありません。同じ文学作品が複数のWorkレコードに分かれている場合もあります。Work IDを選ぶことは、文学作品全体の版を完全に同定することではありません。Author IDの関連Work群は補足的な比較であり、作家の全著作を意味しません。

**年代別グラフ**　2026年2月28日版のEdition出版年の登録を10年単位で集計しています。登録に年がない版や別Workの版もあるため、空白年代は「出版がなかった」とは断定できません。Workに記された初刊年も誤っている場合があります。

**個別Editionの表**　Open Library公開APIの現在の記録から最大20件だけを取得します。出版社・出版地・版の表示・`source_records` に欠損があり得ます。`source_records` はデータの取り込み元であり、所蔵館の一覧ではありません。保存データの全件集計と、公開APIの個別記録を混同しないでください。

**使用したデータ**　Conrad 8作品：Open Library公開ページの保存済み参考値（2026年10月14日）。検索・Work比較・年代・言語：Open Library 2026年2月28日版の講習会用データベース（未監査）。個別Edition：Open Library公開APIの現在のデータ。

**原題表記**　*The Nigger of the “Narcissus”* は1897年の原題を使用していますが、現在では差別的とされる語を含みます。

[Open Library：Joseph Conrad](https://openlibrary.org/authors/OL19441A) · [*Heart of Darkness*](https://openlibrary.org/works/OL38663W) · [Open Library API](https://openlibrary.org/dev/docs/api/books)
